# 04 — R-Tuning (Refusal Training)

Teach the model to say 'I don't know' on out-of-domain or low-confidence questions instead of hallucinating.

**Papers**
- R-Tuning (NAACL 2024) https://arxiv.org/abs/2311.09677
- github.com/shizhediao/R-Tuning

In [ ]:
# ###### Colab bootstrap ######
# On Colab the [experiments] extras pulls both requirements_package.txt and
# requirements_experiments.txt in one pip command — single source of truth lives
# in setup.py's extras_require. Then bootstrap() loads Colab Secrets into
# os.environ and brings up Tailscale so *.ts.net hostnames are reachable.
# Locally bootstrap() just loads .env and checks the tailnet — no installs.
#
# Required Colab Secrets (key icon → Add new secret → toggle "Notebook access"):
#   TAILSCALE_AUTHKEY   – from https://login.tailscale.com/admin/settings/keys
#   MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_INSECURE_TLS
#   MINIO_ENDPOINT / MINIO_ACCESS_KEY / MINIO_SECRET_KEY / MINIO_SECURE
#   HF_TOKEN
import os, subprocess, sys
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "burnit_bg[experiments] @ git+https://github.com/kirilyotov/BurnIT-BG.git",
    ])

from utils.colab import bootstrap
bootstrap()


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name}")
    print(f"VRAM:   {props.total_memory / 1024**3:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")


## 1. Setup & config

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'data_platform').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data_platform.common import set_env
from data_platform.storage import MinioStorage

from experiments.shared.mlflow_utils import setup_run, log_responses, stage, log_training_curves
from experiments.shared.inference_utils import (
    run_full_test_battery,
    TEST_PROMPTS_IN_DOMAIN, TEST_PROMPTS_OUT_OF_DOMAIN, TEST_PROMPTS_EDGE,
    format_prompt,
)
from experiments.shared.model_utils import (
    DEFAULT_MODEL_NAME, load_model_unsloth, apply_qlora, count_trainable_params,
    cuda_device_info,
)
from experiments.shared.eval_utils import compute_perplexity, benchmark_speed, vram_snapshot
from experiments.shared.dataset_utils import (
    load_alpaca_dataset, dataset_statistics, alpaca_to_prompt,
)
from experiments.shared.dataset_browser import list_datasets, pick_dataset, download_dataset, resolve


In [ ]:
set_env(quiet=True)
tracking, tags, run_name = setup_run(
    experiment='r_tuning',
    model=DEFAULT_MODEL_NAME,
    stage="experiment",
    mlflow_experiment_name='burnit-bg-experiments',
)
print(f"run_name = {run_name}")
print(f"tags     = {tags}")
print(f"machine  = {cuda_device_info()}")


## 2. Data loading

In [ ]:
# ###### Pick a dataset from MinIO (or fall back to local data_prep/) ######
# Set DATASET_PREFIX in .env / Colab secrets to skip the picker, e.g.
#   DATASET_PREFIX=data_prep/processed/mental-health
# Or pass `auto=True` below for non-interactive runs.

DEFAULT_PREFIX = os.getenv("DATASET_PREFIX", "data_prep/processed")
try:
    local_dataset_dir = resolve(prefix=DEFAULT_PREFIX, auto=False)
    TRAIN_PATH = local_dataset_dir / "train.jsonl"
    EVAL_PATH  = local_dataset_dir / "eval.jsonl"
except (FileNotFoundError, Exception) as exc:
    print(f"[data] MinIO lookup failed ({exc}); falling back to local data_prep/processed/")
    TRAIN_PATH = REPO_ROOT / "data_prep/processed/train.jsonl"
    EVAL_PATH  = REPO_ROOT / "data_prep/processed/eval.jsonl"

train_records = list(load_alpaca_dataset(TRAIN_PATH))
eval_records  = list(load_alpaca_dataset(EVAL_PATH))
train_stats = dataset_statistics(train_records)
eval_stats  = dataset_statistics(eval_records)
print(f"train: {len(train_records)} rows  eval: {len(eval_records)} rows")
print(train_stats)


## 2b. Load best prior model + dataset

In [ ]:
# Pick the lowest-perplexity prior run (baseline or layer-pruning).
# Set PRIOR_RUN_ID in env to override.
PRIOR_RUN_ID = os.getenv("PRIOR_RUN_ID")
model, tokenizer = load_model_unsloth(DEFAULT_MODEL_NAME, max_seq_length=2048,
                                      load_in_4bit=True, token=os.getenv("HF_TOKEN"))
if PRIOR_RUN_ID:
    try:
        local_path = tracking.load_model(run_id=PRIOR_RUN_ID, artifact_path="model")
        print(f"loaded weights from {local_path}")
    except Exception as exc:
        print(f"[warn] prior model load failed ({exc}); using base.")
model = apply_qlora(model)


## 3. Refusal dataset generation

In [ ]:
# Use the current model to answer each eval prompt; flag wrong answers
# and relabel them as refusals. Then mix in out-of-domain refusals.
import torch
REFUSAL_TEMPLATE = (
    "Не съм сигурен/а за това. Не искам да Ви дам грешна информация. "
    "Бихте ли преформулирали или потърсили професионална помощ?"
)

@torch.no_grad()
def generate(prompt, max_new=128):
    p = format_prompt(prompt, template="alpaca")
    ids = tokenizer(p, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)

# Cheap correctness signal: word-overlap with the gold answer.
def looks_correct(predicted: str, gold: str, threshold=0.3) -> bool:
    p_tokens = set(predicted.lower().split())
    g_tokens = set(gold.lower().split())
    if not g_tokens: return True
    return (len(p_tokens & g_tokens) / len(g_tokens)) >= threshold

new_records = []
flagged = 0
for r in eval_records:
    pred = generate(r["instruction"])
    if looks_correct(pred, r["output"]):
        new_records.append(r)
    else:
        new_records.append({**r, "output": REFUSAL_TEMPLATE, "is_refusal": True})
        flagged += 1

# Append explicit out-of-domain refusals
from experiments.shared.dataset_utils import make_record
OOD = [
    ("Какво е блокчейн?", REFUSAL_TEMPLATE),
    ("Напиши Python код за уеб скрейпинг.", REFUSAL_TEMPLATE),
    ("Кой ще спечели Шампионската лига?", REFUSAL_TEMPLATE),
]
for q, a in OOD:
    new_records.append(make_record(
        instruction=q, output=a, source="synthetic_refusal",
        category="out_of_domain", is_refusal=True, language="bg",
    ))
print(f"flagged {flagged} for refusal-relabeling; total dataset = {len(new_records)}")
refusal_ratio = sum(1 for r in new_records if r.get("is_refusal")) / len(new_records)
tracking.log_metrics({"r_tuning.refusal_ratio": refusal_ratio, "r_tuning.dataset_size": len(new_records)})


## 4. Fine-tune with reweighted refusal loss

In [ ]:
# trl's SFTTrainer doesn't support per-record loss weights directly;
# we approximate it by oversampling refusal records 2x in the training set.
import random
random.seed(42)
oversampled = list(new_records) + [r for r in new_records if r.get("is_refusal")]
random.shuffle(oversampled)

from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

train_ds = Dataset.from_list(oversampled).map(lambda r: {**r, "text": alpaca_to_prompt(r, eos_token=tokenizer.eos_token)})

OUTPUT_DIR = REPO_ROOT / "tmp/experiments/r_tuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), num_train_epochs=2,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=1e-4, logging_steps=10, save_strategy="epoch",
    bf16=True, optim="adamw_8bit", report_to=[], max_grad_norm=1.0,
)
with tracking.run(run_name=run_name, tags=tags):
    tracking.log_params({"r_tuning.oversample_factor": 2})
    with stage(tracking, "train"):
        trainer = SFTTrainer(
            model=model, tokenizer=tokenizer,
            train_dataset=train_ds, dataset_text_field="text",
            max_seq_length=2048, args=training_args,
        )
        trainer.train()


## 5. Calibration evaluation

In [ ]:
from experiments.shared.eval_utils import compute_ece
# Bin "confidence" by output entropy proxy: longer, more-detailed answer = more confident.
@torch.no_grad()
def confidence_and_correct(records, n=64):
    confidences, correctness = [], []
    for r in records[:n]:
        pred = generate(r["instruction"])
        confidence = min(1.0, len(pred.split()) / 50.0)
        correctness.append(looks_correct(pred, r["output"]))
        confidences.append(confidence)
    return confidences, correctness

confs, corrs = confidence_and_correct(eval_records, n=64)
ece = compute_ece(confs, corrs)
tracking.log_metrics({"calibration.ece": float(ece)})
print(f"ECE = {ece:.4f}")


## 6. Inference test

In [ ]:
# ###### Inference test (mental-health prompts) ######
with stage(tracking, "inference_test"):
    batteries = run_full_test_battery((model, tokenizer), max_new_tokens=256)
    log_responses(
        tracking,
        experiment='r_tuning',
        model_checkpoint=str(REPO_ROOT / "tmp/experiments/r_tuning"),
        **batteries,
    )
    for k, v in batteries.items():
        print(f"-- {k} --")
        for entry in v[:2]:
            print(f"  Q: {entry['prompt'][:80]}\n  A: {entry['response'][:200]}\n")


## 7. Save via MLflow + GGUF export

In [ ]:
# ###### Save model via MLflow (single source of truth) ######
# tracking.log_model logs the model artifact under runs:/<id>/model and,
# when registered_model_name is set, adds a new version to the Models tab.
# MLflow's artifact store backs onto MinIO — no separate upload needed.
with stage(tracking, "save_model"):
    try:
        tracking.log_model(
            model,
            flavor="transformers",
            artifact_path="model",
            registered_model_name='burnit-bg-r-tuning',
            input_example=None,
        )
        print("[save] model logged via MLflow")
    except Exception as exc:
        print(f"[save] log_model failed: {exc}")

# Optional: GGUF export for offline local inference (RTX 3050 / Ollama).
# The GGUF is added as a run artifact under `gguf/`.
with stage(tracking, "gguf_export"):
    try:
        from experiments.shared.model_utils import export_gguf
        gguf_path = export_gguf(model, tokenizer, REPO_ROOT / "tmp/experiments/r_tuning/gguf", quantization="q4_k_m")
        tracking.save_data(gguf_path, artifact_path="gguf")
        print(f"[save] GGUF logged: {gguf_path}")
    except Exception as exc:
        print(f"[save] GGUF export skipped: {exc}")
